# Generar reporte Inventario Bogotá - PAI Observatorio

Este notebook genera el archivo de excel con el listado de investigaciones del Plan Anual de Investigaciones de la dog1Dirección Observatorio y Gestión del Conocimiento Cultural. A partir del listado de investigaciones toma las columnas requeridas y disponibles por el formato de Inventario Bogotá y las dispone según lo indicado.

- Por: javier.ojeda@scrd.gov.co
- Fecha: 27/04/2026


In [56]:
import pandas as pd

In [65]:
# Carpeta donde se encuentran los archivos de investigaciones, con las hojas correspondientes a cada año
data_folder = '../../content/data/investigaciones'

# Carpeta ruta local donde se guarda el archivo excel con el reporte de planeación
folder_destino = "D:\\docs\\trabajo\\SecCultura\\PAI 2026\\Otros\\Reporte Planeación"

In [58]:
df_investigaciones = pd.read_json(f'{data_folder}/investigaciones.json')

# Filtrar solo las que tienen "InventarioBogota" en el campo "tags"
mask_inventario_bogota = df_investigaciones['tags'].fillna('').str.contains('InventarioBogota', regex=False)
df_investigaciones = df_investigaciones[mask_inventario_bogota]

In [59]:
# Leer desde Drive archivo CSV con las columnas requeridas para el reporte de InventarioBogota

url_csv = 'https://docs.google.com/spreadsheets/d/1PPHT3cR2GfnDORIa6fPpXfCeJt2IhAbqMSKAxYNNI4U/export?format=csv&gid=1424461837'
df_columnas = pd.read_csv(url_csv)

df_columnas.head()

,columna,descripcion,obligatorio,equivalente_pai
0,1. Título del Estudio,Corresponde al nombre dado al estudio tal como...,Sí,titulo
1,2. Descripción,Resumen o breve explicación del contenido del ...,Sí,descripcion
2,3. Palabras Claves,Temáticas asociadas al estudio o términos que ...,Sí,palabras_clave
3,4. Tipo de Estudio,Identificar de la lista despegable el tipo de ...,Sí,NaN
4,5. Observatorio,En caso de que el estudio sea desarrollado por...,No,NaN


In [60]:
# Crear dataframe con las columnas requeridas, inicializando con valores vacíos
columnas_requeridas = df_columnas['columna'].tolist()
df_inventario_bogota = pd.DataFrame(columns=columnas_requeridas)

In [61]:
# Recorrer df_columnas y llenar df_inventario_bogota con los datos correspondientes de df_investigaciones
for _, row in df_columnas.iterrows():
    columna = row['columna']
    campo_investigaciones = row['equivalente_pai']
    
    if campo_investigaciones in df_investigaciones.columns:
        df_inventario_bogota[columna] = df_investigaciones[campo_investigaciones]

In [62]:
# Campos especiales de InventarioBogota
df_inventario_bogota['4. Tipo de Estudio'] = 'Otro'
df_inventario_bogota['5. Observatorio'] = 'Dirección Observatorio y Gestión del Conocimiento Cultural'
df_inventario_bogota['6. Entidad Distrital'] = 'Secretaría de Cultura, Recreación y Deporte'
df_inventario_bogota['7. Dependencia'] = 'Dirección Observatorio y Gestión del Conocimiento Cultural'
df_inventario_bogota['12. Fuente de financiación'] = 'Secretaría de Cultura, Recreación y Deporte'
df_inventario_bogota['13. Fuentes de información'] = 'Encuestas y mediciones propias'

# Campo de link se concatena con el id del estudio y un link base

base_link = 'https://observatoriocultura.github.io/observatorio/#/investigaciones?investigacion_id='

df_inventario_bogota['16. Link al estudio'] = base_link + df_inventario_bogota['16. Link al estudio'].astype(str)



In [64]:
# Guardar el el df_inventario_bogota en archivo excel ya existente, reemplazando la hoja "estudios"

filename = "2026 SCRD Observatorio de Cultura - Estudios Inventario Bogotá.xlsx"

with pd.ExcelWriter(f'{folder_destino}/{filename}', mode='a', if_sheet_exists='replace') as writer:
    df_inventario_bogota.to_excel(writer, sheet_name='estudios', index=False)